In [22]:
import numpy as np
import pandas as pd
from tensorflow import keras
import seaborn as sns
import matplotlib.pyplot as plt
from keras import layers
from sklearn.preprocessing import StandardScaler
import numpy as np
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

In [32]:
#import the datasets to test
def load_data():
    data = np.load("roads_canids_windows_200hz_3s.npz")
    # Access arrays by their keys
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test  = data["X_test"]
    y_test  = data["y_test"]
    X_train = X_train[..., :-1] #TEMP FIX REMOVE LATER - FIX PREPROCESSING TO GET RID OF STRING COLUMN
    X_test  = X_test[..., :-1] #TEMP FIX REMOVE LATER
    print(X_train.shape)
    print(y_train.shape)
    print(X_test.shape)
    print(y_test.shape)
    data.close()
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = load_data()

(11280, 600, 23)
(11280,)
(13948, 600, 23)
(13948,)


In [33]:
def standardize(X_train, y_train, X_test, y_test):
    # Step 1: Clip outliers (important for ROAD)
    X_train = np.clip(X_train, -1e6, 1e6)
    X_test  = np.clip(X_test,  -1e6, 1e6)
    # Step 2: Standardize features
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, X_train.shape[-1])
    X_test_flat  = X_test.reshape(-1,  X_test.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_test_scaled  = scaler.transform(X_test_flat)
    X_train = X_train_scaled.reshape(X_train.shape)
    X_test  = X_test_scaled.reshape(X_test.shape)
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = standardize(X_train, y_train, X_test, y_test)

In [25]:
import numpy as np
print("Train:", np.unique(y_train, return_counts=True))
print("Test:", np.unique(y_test, return_counts=True))
print(y_train.dtype, y_train[:20])
print("Min/max:", X_train.min(), X_train.max())
print("Std:", X_train.std())
print("Xtrin analysis: ", X_train[0,0])

Train: (array([0, 1]), array([10214,  1066]))
Test: (array([0, 1]), array([13648,   300]))
int64 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Min/max: -188.16494512283168 187.42301713498745
Std: 0.999999999992464
Xtrin analysis:  [-0.91757279 -0.33602709 -0.06113441 -0.37096399 -0.21218677 -0.41802491
 -0.2607396  -0.06516671 -0.51090135 -0.33026307 -0.20267445 -0.59332852
 -0.37099857 -0.41061103 -0.3505285  -1.3405736  -3.00375961 -0.91341512
 -4.29404378 -5.62925525 -1.25825012 -0.09747152 -0.22041024]


In [29]:
def create_model(): #this is the same model we'll always use for all. 
    model = keras.Sequential()
    model.add(layers.Input(shape=(600, 24)))
    model.add(layers.Conv1D(16, 4, activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid')) #output 1 bc we only have 2 labels: attack or not attack
    return model
model = create_model()

In [30]:
#train data 
b_size = 32
callbacks = [
    ModelCheckpoint("saved_models/best_model_road_cnn.keras", monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-9, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=16, verbose=1, restore_best_weights=True)
]
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(X_train, y_train, batch_size = b_size, epochs = 30, validation_split=0.1, callbacks = callbacks, verbose = 1)

Epoch 1/30


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "conv1d_5" is incompatible with the layer: expected axis -1 of input shape to have value 24, but received input with shape (None, 600, 23)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 600, 23), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
testing_acc = model.evaluate(X_test,y_test, verbose=1)
print(f"Test loss: {testing_acc[0]}")
print(f"Test accuracy: {testing_acc[1]}")

436/436 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9785 - loss: 0.0000e+00
Test loss: 0.0
Test accuracy: 0.9784915447235107
